In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip uninstall -y datasets

Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5


In [3]:
!pip install datasets==2.17

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 4.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 3.0 MB/s eta 0:00:0000:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

In [4]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Load dataset
dataset_fabsa = load_dataset("jordiclive/fabsa")
train_ds = dataset_fabsa["train"]
test_ds = dataset_fabsa["test"]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/747k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [6]:
# Extract unique Aspect Labels (ignoring sentiment)
all_aspects = set()

# We iterate over the raw list to avoid tensor errors
for label_entry in train_ds['labels']:
    for aspect, sentiment in label_entry:
        all_aspects.add(aspect)

aspect_list = sorted(list(all_aspects))
num_labels = len(aspect_list)
label2id = {l: i for i, l in enumerate(aspect_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Found {num_labels} unique categories: {aspect_list}")

Found 12 unique categories: ['Account management: Account access', 'Company brand: Competitor', 'Company brand: General satisfaction', 'Company brand: Reviews', 'Logistics rides: Speed', 'Online experience: App website', 'Purchase booking experience: Ease of use', 'Staff support: Attitude of staff', 'Staff support: Email', 'Staff support: Phone', 'Value: Discounts promotions', 'Value: Price value for money']


In [7]:
# Data Processing (Multi-Hot Encoding)
def encode_data(example):
    # Create a vector of zeros [0, 0, ... 0]
    vec = [0.0] * num_labels 
    
    # Loop through labels, get aspect, ignore sentiment
    for aspect, sentiment in example["labels"]:
        if aspect in label2id:
            idx = label2id[aspect]
            vec[idx] = 1.0
            
    # Tokenize text
    enc = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    
    # Add labels to the encoding
    enc["labels"] = vec
    return enc

In [8]:
# Initialize Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Apply processing
train_ds = train_ds.map(encode_data, batched=False)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Apply processing
test_ds = test_ds.map(encode_data, batched=False)
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [10]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [11]:
# Load model
bert = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Apply LoRA
lora_cfg = LoraConfig(
    r=16,          # Rank (Paper uses full fine-tuning, but r=16 is good for LoRA)
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

In [13]:
model = get_peft_model(bert, lora_cfg)
model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): L

In [14]:
# Optimizer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [15]:
loss_fn = nn.BCEWithLogitsLoss()

In [16]:
epochs = 10 # FABSA paper suggests training until convergence (usually 5-10 epochs)

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].float().to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")


Starting Training...


Epoch 1: 100%|██████████| 496/496 [01:58<00:00,  4.19it/s]


Epoch 1 Loss: 0.3390


Epoch 2: 100%|██████████| 496/496 [02:10<00:00,  3.79it/s]


Epoch 2 Loss: 0.2770


...

Epoch 8 Loss: 0.1607


Epoch 9: 100%|██████████| 496/496 [02:10<00:00,  3.81it/s]


Epoch 9 Loss: 0.1528


Epoch 10: 100%|██████████| 496/496 [02:10<00:00,  3.81it/s]

Epoch 10 Loss: 0.1450


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

model.eval()
y_true = []
y_pred = []

print("\nStarting Evaluation...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Sigmoid activation -> Probability
        probs = torch.sigmoid(logits)

        # Threshold
        preds = (probs > 0.3).int().cpu().numpy()
        labels = batch["labels"].cpu().numpy()

        y_true.extend(labels)
        y_pred.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Macro Metrics
macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

# Weighted Metrics
weighted_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
weighted_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nMacro Metrics")
print(f"Precision: {macro_precision:.4f}")
print(f"Recall:    {macro_recall:.4f}")
print(f"F1-score:  {macro_f1:.4f}")

print("\nWeighted Metrics")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall:    {weighted_recall:.4f}")
print(f"F1-score:  {weighted_f1:.4f}")


Starting Evaluation...


100%|██████████| 100/100 [00:12<00:00,  8.03it/s]



Macro Metrics
Precision: 0.6472
Recall:    0.7444
F1-score:  0.6903

Weighted Metrics
Precision: 0.7151
Recall:    0.8277
F1-score:  0.7661


In [18]:
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

sample_accuracies = []
for t, p in zip(y_true_arr, y_pred_arr):
    correct = (t * p).sum()               # count correctly predicted labels
    total = t.sum()                       # total actual labels for that sample
    if total == 0:                         # if no true labels exist
        sample_accuracies.append(1.0)      
    else:
        sample_accuracies.append(correct / total)

overall_label_accuracy = np.mean(sample_accuracies)
print(f"Label-wise Sample Accuracy: {overall_label_accuracy:.4f}")


Label-wise Sample Accuracy: 0.8543
